In [2]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 11.4 MB/s eta 0:00:00


In [3]:
import keras_tuner
import keras

In [4]:
import zipfile
import csv
import io

# Open the zip file and extract the CSV data
zip_file_path = '/content/archive (1) (2).zip'
csv_file_name = 'test.csv'

data = []  # Initialize an empty list to store the data

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    with zip_ref.open(csv_file_name) as csv_file:
        # Read the CSV file contents and convert them into tuples
        csv_data = csv.reader(io.TextIOWrapper(csv_file, 'utf-8'))
        next(csv_data)  # Skip header if exists
        for row in csv_data:
            text = row[0]  # Assuming text is in the first column
            sentiment = row[1]  # Assuming sentiment is in the second column
            # Create a tuple with text and sentiment and append it to the data list
            data.append((text, sentiment))


In [8]:
# Import necessary libraries
import numpy as np

from keras.src.layers import Bidirectional  # Import Bidirectional layer from Keras
from keras_tuner import HyperModel, BayesianOptimization  # Import HyperModel and BayesianOptimization from Keras Tuner
from tensorflow.keras.models import Sequential  # Import Sequential model from TensorFlow Keras
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Conv1D, GlobalMaxPooling1D  # Import various layers from TensorFlow Keras
from tensorflow.keras.preprocessing.text import Tokenizer  # Import Tokenizer for text preprocessing
from tensorflow.keras.preprocessing.sequence import pad_sequences  # Import pad_sequences for padding sequences
from tensorflow.keras.optimizers import Adam  # Import Adam optimizer
from tensorflow.keras.losses import SparseCategoricalCrossentropy  # Import SparseCategoricalCrossentropy loss
from sklearn.utils.class_weight import compute_class_weight  # Import compute_class_weight for calculating class weights
from sklearn.model_selection import train_test_split  # Import train_test_split for splitting data
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau  # Import callbacks for model training
from keras_tuner.tuners import RandomSearch  # Import RandomSearch tuner from Keras Tuner
from tensorflow.keras.losses import SparseCategoricalCrossentropy  # Import SparseCategoricalCrossentropy loss from TensorFlow Keras
from keras_tuner.engine.hypermodel import HyperModel  # Import HyperModel from Keras Tuner
from nltk.corpus import stopwords  # Import stopwords from NLTK corpus
import re  # Import re module for regular expressions
import nltk  # Import NLTK library for natural language processing
print("Imported")
# Download NLTK stopwords
nltk.download('stopwords')


 # Contains text samples and their corresponding sentiment labels

# Preprocessing function to clean and tokenize text data
def preprocess(text):
    text = text.lower()  # Convert text to lowercase
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    text = " ".join([word for word in text.split() if word not in stopwords.words('english')])  # Remove stopwords
    return text

# Apply preprocessing to the dataset


corpus = [preprocess(text) for text,_ in data]

# Tokenization and padding
tokenizer = Tokenizer(num_words=5000, oov_token='<OOV>')  # Initialize Tokenizer with a vocabulary size limit
tokenizer.fit_on_texts(corpus)  # Fit Tokenizer on preprocessed text data
word_index = tokenizer.word_index  # Get word index from Tokenizer
sequences = tokenizer.texts_to_sequences(corpus)  # Convert text to sequences of indices
padded_sequences = pad_sequences(sequences, padding='post')  # Pad sequences to the same length

# Convert sentiment labels to numerical values
sentiment_mapping = {"positive": 0, "negative": 1, "neutral": 2,"neg": 1,"pos": 0,"neu":2}  # Mapping of sentiment labels to numerical values
labels = np.array([sentiment_mapping[sentiment] for _, sentiment in data])  # Convert sentiment labels to numerical values

# Split the data into train and validation sets
train_texts, val_texts, train_labels, val_labels = train_test_split(padded_sequences, labels, test_size=0.2, random_state=42)  # Split data into training and validation sets

# Calculate class weights for imbalanced dataset
class_counts = np.bincount(train_labels)  # Count the occurrences of each class in the training labels
total_samples = sum(class_counts)  # Calculate the total number of samples
class_weights = {cls: total_samples / count for cls, count in enumerate(class_counts)}  # Calculate class weights

# Define a custom hypermodel for Keras Tuner
class SentimentHyperModel(HyperModel):
    def build(self, hp):
        # Build the model architecture with hyperparameters
        model = Sequential()  # Initialize a Sequential model
        model.add(Embedding(len(word_index) + 1, hp.Int('embedding_dim', min_value=64, max_value=256, step=32)))  # Add an Embedding layer
        model.add(Bidirectional(LSTM(hp.Int('lstm_units', min_value=64, max_value=128, step=32), return_sequences=True)))  # Add a Bidirectional LSTM layer
        model.add(GlobalMaxPooling1D())  # Add a GlobalMaxPooling1D layer

        model.add(Dense(hp.Int('dense_units', min_value=64, max_value=256, step=32), activation='relu'))  # Add a Dense layer with ReLU activation
        model.add(Dropout(hp.Float('dense_dropout', min_value=0.2, max_value=0.5, step=0.1)))  # Add a Dropout layer

        model.add(Dense(3, activation='softmax'))  # Add a Dense layer with softmax activation
        optimizer = Adam(learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1, sampling='LOG'))  # Initialize Adam optimizer with a learning rate
        loss = SparseCategoricalCrossentropy()  # Use SparseCategoricalCrossentropy loss
        model.compile(loss=loss, optimizer=optimizer, metrics=['accuracy'])  # Compile the model
        return model

# Instantiate the custom hypermodel
hypermodel = SentimentHyperModel()

# Use Bayesian optimization for hyperparameter tuning
tuner = RandomSearch(
    hypermodel,
    objective='val_accuracy',
    max_trials=10,
    directory='my_dir',
    project_name='sentiment_analysis'
)

# Perform the hyperparameter search using training data
tuner.search(train_texts, train_labels, epochs=4, validation_data=(val_texts, val_labels),
             class_weight=class_weights, callbacks=[
        EarlyStopping(patience=3, restore_best_weights=True),
        ReduceLROnPlateau(factor=0.2, patience=2)
    ])

# Load and use the best hyperparameters found by the tuner
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

# Build the model using the best hyperparameters
best_model = hypermodel.build(best_hps)
print(best_model)

# Train the best model with your training data
best_model.fit(train_texts, train_labels, epochs=4, validation_data=(val_texts, val_labels),#epochs=100
               class_weight=class_weights)

# Function to predict sentiment using the trained best model
def predict_sentiment_with_best_model(user_input, model):
    preprocessed_input = preprocess(user_input)  # Preprocess user input
    sequence = tokenizer.texts_to_sequences([preprocessed_input])  # Convert preprocessed input to sequences
    padded_sequence = pad_sequences(sequence, padding='post', maxlen=100)  # Pad the sequence
    sentiment_probabilities = model.predict(padded_sequence)[0]  # Predict sentiment probabilities
    predicted_sentiment = np.argmax(sentiment_probabilities)  # Get the index of the predicted sentiment
    return list(sentiment_mapping.keys())[list(sentiment_mapping.values()).index(predicted_sentiment)]  # Map index to sentiment label

# Load the best model after tuning
best_model = tuner.get_best_models(num_models=1)[0]

# User interaction loop for sentiment prediction
while True:
    user_input = input("Enter your message (or 'exit' to quit): ")  # Get user input
    if user_input.lower() == 'exit':  # Check if user wants to exit
        break
    sentiment = predict_sentiment_with_best_model(user_input, best_model)  # Predict sentiment
    print("Predicted sentiment:", sentiment)  # Print predicted sentiment


Trial 10 Complete [00h 05m 09s]
val_accuracy: 0.8827999830245972

Best val_accuracy So Far: 0.895799994468689
Total elapsed time: 00h 46m 50s
<Sequential name=sequential_1, built=False>
Epoch 1/4
625/625 ━━━━━━━━━━━━━━━━━━━━ 66s 101ms/step - accuracy: 0.8256 - loss: 0.7547 - val_accuracy: 0.8808 - val_loss: 0.2879
Epoch 2/4
625/625 ━━━━━━━━━━━━━━━━━━━━ 63s 100ms/step - accuracy: 0.9165 - loss: 0.4353 - val_accuracy: 0.8824 - val_loss: 0.2765
Epoch 3/4
625/625 ━━━━━━━━━━━━━━━━━━━━ 63s 100ms/step - accuracy: 0.9454 - loss: 0.2954 - val_accuracy: 0.8770 - val_loss: 0.3028
Epoch 4/4
625/625 ━━━━━━━━━━━━━━━━━━━━ 63s 101ms/step - accuracy: 0.9590 - loss: 0.2239 - val_accuracy: 0.8806 - val_loss: 0.3511


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 24 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Enter your message (or 'exit' to quit): winsome performances, several terrific emotional moments, and soulful music are its strengths, the second half could've been sharper
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 258ms/step
Predicted sentiment: positive
Enter your message (or 'exit' to quit): exit
